In [ ]:
"""
Phase 2 - Preprocessing (PaySim Fraud Detection)
Follows exact spec:
1. Load train.csv / validation.csv
2-3. Feature engineering + drop ID columns
4. Encode type with LabelEncoder -> save label_encoder.joblib
5. Train/val split (80/20, stratified) from train.csv
6. validation.csv reserved as final test set (untouched until model selection)
7. Scale numerical columns with StandardScaler -> save scaler.joblib
8. Save X_train, y_train, X_val, y_val, X_test, y_test as CSVs
"""

import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# ============================================================
# 1. Load datasets
# ============================================================
train_df = pd.read_csv("data/raw/train.csv")
test_df = pd.read_csv("data/raw/validation.csv")   # this is the FINAL untouched test set

print("train.csv shape:", train_df.shape)
print("validation.csv (test set) shape:", test_df.shape)


def engineer_features(df):
    df = df.copy()

    # ---- isMerchant: derived BEFORE dropping nameDest ----
    df["isMerchant"] = df["nameDest"].str.startswith("M").astype(int)

    # ---- Balance diff features ----
    df["balanceOrigDiff"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
    df["balanceDestDiff"] = df["newbalanceDest"] - df["oldbalanceDest"]

    # ---- Error balance features (anti-leakage signal) ----
    df["errorBalanceOrig"] = df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
    df["errorBalanceDest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]

    # ---- Drop ID columns now that isMerchant is derived ----
    df = df.drop(columns=["nameOrig", "nameDest"])

    return df


# ============================================================
# 2-5. Feature engineering + drop IDs (applied to both sets)
# ============================================================
train_df = engineer_features(train_df)
test_df = engineer_features(test_df)

# ============================================================
# 3. Encode transaction type with LabelEncoder
# ============================================================
label_encoder = LabelEncoder()
train_df["type"] = label_encoder.fit_transform(train_df["type"])

# Apply the SAME fitted encoder to the test set (never re-fit on test data)
test_df["type"] = label_encoder.transform(test_df["type"])

joblib.dump(label_encoder, "models/label_encoder.joblib")
print("\nSaved label_encoder.joblib")
print("Type classes:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

# ============================================================
# 6. Train / Validation split (80/20, stratified) from train.csv
# ============================================================
X = train_df.drop(columns=["isFraud"])
y = train_df["isFraud"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print("\nX_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("Train fraud rate:", y_train.mean())
print("Val fraud rate:", y_val.mean())

# ============================================================
# 7. Test dataset (validation.csv) — held out, used only at final evaluation
# ============================================================
X_test = test_df.drop(columns=["isFraud"])
y_test = test_df["isFraud"]

print("\nX_test shape:", X_test.shape)
print("Test fraud rate:", y_test.mean())

# ============================================================
# 8. Scale numerical columns with StandardScaler
# ============================================================
numeric_cols = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "balanceOrigDiff",
    "balanceDestDiff",
    "errorBalanceOrig",
    "errorBalanceDest",
]

scaler = StandardScaler()

# Fit ONLY on X_train, transform all three sets
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val[numeric_cols] = scaler.transform(X_val[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

joblib.dump(scaler, "models/scaler.joblib")
print("\nSaved scaler.joblib")

# ============================================================
# 9. Save final datasets
# ============================================================
X_train.to_csv("data/processed/X_train.csv", index=False)
y_train.to_csv("data/processed/y_train.csv", index=False)

X_val.to_csv("data/processed/X_val.csv", index=False)
y_val.to_csv("data/processed/y_val.csv", index=False)

X_test.to_csv("data/processed/X_test.csv", index=False)
y_test.to_csv("data/processed/y_test.csv", index=False)

print("\n============================================================")
print("PREPROCESSING COMPLETE")
print("============================================================")
print("Saved to data/processed/:")
print(" - X_train.csv, y_train.csv")
print(" - X_val.csv, y_val.csv")
print(" - X_test.csv, y_test.csv")
print("\nSaved to models/:")
print(" - label_encoder.joblib")
print(" - scaler.joblib")
print("\nFinal feature columns:", list(X_train.columns))
print("\nNext step: train_model.py")